In [1]:
# imports
import time
import pandas as pd
import requests
from datetime import datetime
from riotwatcher import LolWatcher, ApiError
from pathlib import Path

# repository root: if running as a script __file__ is inside src/, go up one more.
# If running as a notebook, cwd is already src/, so parents[0] is the repo root.
try:
    ROOT = Path(__file__).resolve().parents[1]
except NameError:
    ROOT = Path().resolve().parents[0]


In [ ]:
import time
import requests
import pandas as pd
from datetime import datetime
from pathlib import Path
from riotwatcher import LolWatcher, ApiError

try:
    ROOT = Path(__file__).resolve().parents[1]
except NameError:
    ROOT = Path().resolve().parents[0]

REGION = "americas"
PLATFORM = "na1"
OUT_DIR = ROOT / "data" / "match_data"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── API key ────────────────────────────────────────────────────────────────────
def load_key():
    for p in [
        Path.cwd() / "riot_api_key.txt",
        Path.cwd() / "src" / "riot_api_key.txt",
        ROOT / "riot_api_key.txt",
    ]:
        if p.exists():
            return p.read_text().strip()
    raise FileNotFoundError("riot_api_key.txt not found.")

api_key = load_key()
watcher = LolWatcher(api_key)
print(f"Loaded API key: {api_key[:10]}... (length: {len(api_key)})")

# ── Sentinel for key expiry ────────────────────────────────────────────────────
class KeyExpiredError(Exception):
    pass

# ── Parsing ────────────────────────────────────────────────────────────────────
def parse_match(match_json):
    meta = match_json.get("metadata", {})
    info = match_json.get("info", {})
    match_id = meta.get("matchId") or match_json.get("gameId")

    team_map = {}
    for t in info.get("teams", []):
        tid = t.get("teamId")
        obj = t.get("objectives", {})
        team_map[tid] = {
            "team_win":               t.get("win", False),
            "team_baron_kills":       obj.get("baron",      {}).get("kills", 0),
            "team_dragon_kills":      obj.get("dragon",     {}).get("kills", 0),
            "team_tower_kills":       obj.get("tower",      {}).get("kills", 0),
            "team_inhibitor_kills":   obj.get("inhibitor",  {}).get("kills", 0),
            "team_rift_herald_kills": obj.get("riftHerald", {}).get("kills", 0),
        }

    rows = []
    for p in info.get("participants", []):
        tid = p.get("teamId")
        team = team_map.get(tid, {})
        kills, deaths, assists = p.get("kills", 0), p.get("deaths", 0), p.get("assists", 0)
        rows.append({
            "match_id":        match_id,
            "puuid":           p.get("puuid"),
            "summonerName":    p.get("riotIdGameName") or p.get("summonerName"),
            "championName":    p.get("championName"),
            "championId":      p.get("championId"),
            "teamId":          tid,
            "teamPosition":    p.get("teamPosition"),
            "role":            p.get("role"),
            "win":             p.get("win"),
            "kills":           kills,
            "deaths":          deaths,
            "assists":         assists,
            "kda":             (kills + assists) / max(1, deaths),
            "totalDamageToChampions":    p.get("totalDamageDealtToChampions", 0),
            "physicalDamageToChampions": p.get("physicalDamageDealtToChampions", 0),
            "magicDamageToChampions":    p.get("magicDamageDealtToChampions", 0),
            "totalDamageTaken":          p.get("totalDamageTaken", 0),
            "damageSelfMitigated":       p.get("damageSelfMitigated", 0),
            "damageToBuildings":         p.get("damageDealtToBuildings", 0),
            "damageToObjectives":        p.get("damageDealtToObjectives", 0),
            "goldEarned":              p.get("goldEarned", 0),
            "goldSpent":               p.get("goldSpent", 0),
            "totalMinionsKilled":      p.get("totalMinionsKilled", 0),
            "neutralMinionsKilled":    p.get("neutralMinionsKilled", 0),
            "cs":                      p.get("totalMinionsKilled", 0) + p.get("neutralMinionsKilled", 0),
            "visionScore":             p.get("visionScore", 0),
            "wardsPlaced":             p.get("wardsPlaced", 0),
            "wardsKilled":             p.get("wardsKilled", 0),
            "visionWardsBought":       p.get("visionWardsBoughtInGame", 0),
            "timeCCingOthers":         p.get("timeCCingOthers", 0),
            "totalTimeCCDealt":        p.get("totalTimeCCDealt", 0),
            "champLevel":              p.get("champLevel"),
            "doubleKills":             p.get("doubleKills", 0),
            "tripleKills":             p.get("tripleKills", 0),
            "quadraKills":             p.get("quadraKills", 0),
            "pentaKills":              p.get("pentaKills", 0),
            "turretKills":             p.get("turretKills", 0),
            "firstBloodKill":          p.get("firstBloodKill", False),
            "firstBloodAssist":        p.get("firstBloodAssist", False),
            "longestTimeSpentLiving":  p.get("longestTimeSpentLiving"),
            "totalTimeSpentDead":      p.get("totalTimeSpentDead"),
            "spell1Casts":             p.get("spell1Casts", 0),
            "spell2Casts":             p.get("spell2Casts", 0),
            "spell3Casts":             p.get("spell3Casts", 0),
            "spell4Casts":             p.get("spell4Casts", 0),
            "team_win":                team.get("team_win"),
            "team_baron_kills":        team.get("team_baron_kills", 0),
            "team_dragon_kills":       team.get("team_dragon_kills", 0),
            "team_tower_kills":        team.get("team_tower_kills", 0),
            "team_inhibitor_kills":    team.get("team_inhibitor_kills", 0),
            "team_rift_herald_kills":  team.get("team_rift_herald_kills", 0),
        })
    return rows

# ── Fetch helpers ──────────────────────────────────────────────────────────────
def fetch_match(region, mid, total_timeout=120):
    start = time.time()
    while time.time() - start < total_timeout:
        try:
            return watcher.match.by_id(region, mid)
        except ApiError as e:
            code = e.response.status_code
            if code == 401:
                raise KeyExpiredError()
            elif code == 429:
                wait = int(e.response.headers.get("Retry-After", 2))
                print(f"  Rate limit — waiting {wait}s...")
                time.sleep(wait + 1)
            else:
                print(f"  API error {code} for {mid}, retrying...")
                time.sleep(1)
        except KeyExpiredError:
            raise
        except Exception as e:
            print(f"  Error fetching {mid}: {e}")
            time.sleep(1)
    print(f"  Timed out on {mid}, skipping.")
    return None

def get_match_ids(puuid, region, max_matches=None):
    ids, start = [], 0
    while True:
        try:
            batch = watcher.match.matchlist_by_puuid(region, puuid, start=start, count=100)
        except ApiError as e:
            if e.response.status_code == 401:
                raise KeyExpiredError()
            elif e.response.status_code == 429:
                wait = int(e.response.headers.get("Retry-After", 2))
                time.sleep(wait + 1)
                continue
            break
        if not batch:
            break
        ids.extend(batch)
        if max_matches and len(ids) >= max_matches:
            ids = ids[:max_matches]
            break
        start += 100
        time.sleep(0.4)
    return ids

# ── Main collection ────────────────────────────────────────────────────────────
def collect(region=REGION, platform=PLATFORM, max_players=None, max_matches_per_player=None,
            out_file="challenger_matches.csv", save_every=1000, print_every=10):
    out_path = OUT_DIR / out_file

    if out_path.exists():
        existing = pd.read_csv(out_path)
        all_rows = existing.to_dict("records")
        seen = set(existing["match_id"].dropna().unique())
        print(f"Resuming from {out_path.name}: {len(seen)} matches already collected ({len(all_rows)} rows)")
    else:
        all_rows, seen = [], set()
        print(f"Starting fresh → {out_path}")

    try:
        r = requests.get(
            f"https://{platform}.api.riotgames.com/lol/league/v4/challengerleagues/by-queue/RANKED_SOLO_5x5",
            headers={"X-Riot-Token": api_key},
        )
        r.raise_for_status()
    except Exception as e:
        print(f"Failed to fetch Challenger list: {e}")
        return None

    entries = r.json().get("entries", [])
    print(f"Fetched {len(entries)} Challenger players")
    if max_players:
        entries = entries[:max_players]

    def save():
        pd.DataFrame(all_rows).to_csv(out_path, index=False)

    for i, entry in enumerate(entries, 1):
        puuid = entry.get("puuid")
        if not puuid:
            try:
                summ = watcher.summoner.by_id(platform, entry["summonerId"])
                puuid = summ["puuid"]
                time.sleep(0.4)
            except KeyExpiredError:
                save()
                print(f"\nKey expired resolving player {i}. Saved {len(seen)} matches.")
                print(f"Update riot_api_key.txt then re-run: collect(out_file='{out_file}')")
                return None
            except Exception as e:
                print(f"Player {i}: couldn't resolve puuid — {e}")
                continue

        print(f"\nPlayer {i}/{len(entries)}: {puuid[:12]}...")

        try:
            match_ids = get_match_ids(puuid, region, max_matches=max_matches_per_player)
        except KeyExpiredError:
            save()
            print(f"\nKey expired fetching match IDs for player {i}. Saved {len(seen)} matches.")
            print(f"Update riot_api_key.txt then re-run: collect(out_file='{out_file}')")
            return None

        new_ids = [m for m in match_ids if m not in seen]
        print(f"  {len(match_ids)} match IDs, {len(new_ids)} new")
        if not new_ids:
            continue

        for j, mid in enumerate(new_ids, 1):
            try:
                data = fetch_match(region, mid)
            except KeyExpiredError:
                save()
                print(f"\nKey expired on match {mid}. Saved {len(seen)} matches.")
                print(f"Update riot_api_key.txt then re-run: collect(out_file='{out_file}')")
                return None

            if not data:
                continue
            try:
                rows = parse_match(data)
                all_rows.extend(rows)
                seen.add(mid)
            except Exception as e:
                print(f"  Parse error {mid}: {e}")
            time.sleep(0.4)

            total = len(seen)
            if total % print_every == 0:
                print(f"  [{total} matches] player {i}, match {j}/{len(new_ids)}: {mid}")
            if total % save_every == 0:
                save()
                print(f"  ── saved {total} matches ──")

        time.sleep(1)

    save()
    df = pd.DataFrame(all_rows)
    print(f"\nDone. {len(seen)} matches, {len(df)} participant rows → {out_path}")
    return df

# ── Run ────────────────────────────────────────────────────────────────────────
df = collect(max_players=300, out_file="challenger_matches.csv")


Loaded API key: RGAPI-2673... (length: 42)
Starting fresh → C:\Users\Laser\cs163-group18-winfactors\data\match_data\challenger_matches.csv
Fetched 300 Challenger players

Player 1/300: SyHGed9gT8bx...


In [ ]:
# Test the API key with a simple request
# Run this cell to verify your Riot API key works before running the main pipeline.

import requests

try:
    api_key = load_key()
    print(f"Testing key: {api_key[:10]}... (length: {len(api_key)})")
    headers = {"X-Riot-Token": api_key}
    # Test with a known summoner (change 'Doublelift' to any valid name if needed)
    test_url = "https://na1.api.riotgames.com/lol/summoner/v4/summoners/by-name/Doublelift"
    r = requests.get(test_url, headers=headers)
    print(f"Response status: {r.status_code}")
    if r.status_code == 200:
        data = r.json()
        print(f"Success! Summoner: {data.get('name')}, Level: {data.get('summonerLevel')}")
    else:
        print(f"Failed: {r.text}")
except Exception as e:
    print(f"Error testing key: {e}")